# Explainability (SHAP) untuk Isolation Forest BMKG

Notebook ini **tidak melatih ulang model**. Model (`isolation_forest_bmkg.pkl`) dan scaler (`scaler_isolation_forest_bmkg.pkl`) yang dipakai persis sama dengan yang dihasilkan oleh `isolation_forest.ipynb`, begitu juga dataset historisnya (`../data/bmkg/gabungan_2008_2025.csv`, difilter bounding box Indonesia + dropna, fitur `['mag', 'depth', 'latitude', 'longitude']`, distandarisasi dengan `StandardScaler`).

Pendekatan yang dipakai: **Permutation SHAP** (`shap.PermutationExplainer`), bukan `TreeExplainer`. Alasannya: `TreeExplainer` dirancang untuk model tree-based yang skornya adalah agregasi nilai leaf yang additive terhadap fitur (mis. regresi/klasifikasi Random Forest/Gradient Boosting). `IsolationForest` scikit-learn tidak menghasilkan skor seperti itu — skornya berbasis *path length* (rata-rata kedalaman titik ke daun tempat ia terisolasi, dinormalisasi terhadap `c(n)`), dan SHAP tidak mendukung dekomposisi resmi untuk semantik ini di `TreeExplainer`.

Permutation SHAP valid dipakai karena secara matematis Shapley value hanya butuh satu syarat: adanya fungsi skor `f(x)` yang mengembalikan skalar untuk input `x` — di sini `f(x) = model.decision_function(x)`. Definisi aksiomatik Shapley value (efficiency, symmetry, dummy, additivity) tidak bergantung pada struktur internal model, sehingga valid untuk model unsupervised apa pun, termasuk Isolation Forest. Karena hanya ada 4 fitur, permutation dihitung **exact** (4! = 24 kombinasi), bukan sekadar sampling acak — jadi hasilnya bukan aproksimasi kasar.

## 1. Environment Setup

In [ ]:
# pip install shap>=0.44,<1.0   -> kompatibel dengan scikit-learn>=1.3 (termasuk versi terbaru 1.8.x)

import pickle
import numpy as np
import pandas as pd
import shap

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
FEATURES = ['mag', 'depth', 'latitude', 'longitude']

print(f"SHAP version : {shap.__version__}")

## 2. Load Data Historis (identik dengan isolation_forest.ipynb)

In [ ]:
# Path relatif dari folder "Anomaly Detection BMKG" ke folder "data" -- sama seperti notebook utama
file_path = '../data/bmkg/gabungan_2008_2025.csv'

df = pd.read_csv(file_path)
print(f"Dataset berhasil dimuat dengan dimensi: {df.shape}")

# Bounding box Indonesia
lat_min, lat_max = -11.0, 6.0
lon_min, lon_max = 95.0, 141.0

df = df[(df['latitude'] >= lat_min) & (df['latitude'] <= lat_max) &
        (df['longitude'] >= lon_min) & (df['longitude'] <= lon_max)]
df = df.dropna()
df = df.reset_index(drop=True)

print(f"Jumlah data setelah filter wilayah Indonesia + dropna: {len(df)}")
df.head()

## 3. Load Model & Scaler dari .pkl (TANPA retrain)

In [ ]:
with open('models/isolation_forest_bmkg.pkl', 'rb') as f:
    final_iso_forest = pickle.load(f)

with open('models/scaler_isolation_forest_bmkg.pkl', 'rb') as f:
    scaler = pickle.load(f)

print("Model dan scaler berhasil dimuat dari file .pkl (tidak ada retraining).")
print(final_iso_forest)

## 4. Reproduksi Label & Score Anomali (untuk validasi konsistensi dengan notebook utama)

In [ ]:
X = df[FEATURES]
X_scaled = scaler.transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES)

df['anomaly_score'] = final_iso_forest.decision_function(X_scaled_df)
raw_label = final_iso_forest.predict(X_scaled_df)  # 1 = normal, -1 = anomali (bawaan sklearn)
df['anomaly_label'] = pd.Series(raw_label).map({1: 0, -1: 1})  # 0 = Normal, 1 = Anomali

print(df['anomaly_label'].value_counts().rename(index={0: "Normal (0)", 1: "Anomali (1)"}))

## 5. Background/Reference Dataset untuk SHAP

In [ ]:
# Background dataset: sampel representatif dari data training yang sudah di-scale.
# random_state konsisten -> reproducible.
background = X_scaled_df.sample(n=200, random_state=RANDOM_STATE)

print(f"Background dataset untuk SHAP: {background.shape[0]} baris x {background.shape[1]} fitur")

## 6. Membuat SHAP Explainer (Permutation, exact untuk 4 fitur)

In [ ]:
# Fungsi skor yang dijelaskan: decision_function
# (di scikit-learn: skor POSITIF = normal, skor NEGATIF = anomali)
def score_fn(X_array):
    X_df = pd.DataFrame(X_array, columns=FEATURES)
    return final_iso_forest.decision_function(X_df)

explainer = shap.PermutationExplainer(
    score_fn,
    background,
    seed=RANDOM_STATE
)

print("Explainer siap. Dengan 4 fitur, permutation dihitung secara exact (24 permutasi), bukan sampling acak.")

## 7. Fungsi `explain_anomaly`

In [ ]:
def explain_anomaly(magnitude, depth, latitude, longitude, top_k=None):
    """
    Menjelaskan skor anomali satu titik gempa baru menggunakan SHAP (Permutation Explainer).

    Returns dict:
      - anomaly_score    : skor decision_function asli (makin negatif = makin anomali)
      - is_anomaly       : bool, hasil model.predict()
      - shap_values      : dict fitur -> shap value
                            (positif = menahan/melemahkan anomali, negatif = memperkuat anomali,
                             karena decision_function: positif=normal, negatif=anomali)
      - dominant_feature  : fitur dengan |shap value| terbesar
      - ranking          : list (fitur, shap_value) terurut dari paling berpengaruh -> paling kecil
    """
    raw_input = pd.DataFrame([[magnitude, depth, latitude, longitude]], columns=FEATURES)
    scaled_input = pd.DataFrame(scaler.transform(raw_input), columns=FEATURES)

    anomaly_score = final_iso_forest.decision_function(scaled_input)[0]
    is_anomaly = final_iso_forest.predict(scaled_input)[0] == -1

    shap_result = explainer(scaled_input)
    shap_values = dict(zip(FEATURES, shap_result.values[0]))

    ranking = sorted(shap_values.items(), key=lambda kv: abs(kv[1]), reverse=True)
    dominant_feature = ranking[0][0]

    result = {
        "anomaly_score": float(anomaly_score),
        "is_anomaly": bool(is_anomaly),
        "shap_values": {k: float(v) for k, v in shap_values.items()},
        "dominant_feature": dominant_feature,
        "ranking": [(k, float(v)) for k, v in ranking],
    }

    if top_k is not None:
        result["ranking"] = result["ranking"][:top_k]

    return result


# --- Contoh pemakaian ---
contoh = explain_anomaly(magnitude=6.2, depth=520, latitude=-2.5, longitude=122.0)
contoh

## 8. Precompute SHAP untuk Seluruh Histori Anomali (artefak untuk backend FastAPI)

Backend produksi cukup membaca file hasil precompute ini (JSON/PKL) tanpa perlu meng-install `shap` sama sekali.

In [ ]:
anomaly_rows = df[df['anomaly_label'] == 1].copy()

precomputed = []
for idx, row in anomaly_rows.iterrows():
    exp = explain_anomaly(row['mag'], row['depth'], row['latitude'], row['longitude'])
    precomputed.append({
        "index": idx,
        "time": row['time'],
        "mag": row['mag'],
        "depth": row['depth'],
        "latitude": row['latitude'],
        "longitude": row['longitude'],
        "wilayah": row['wilayah'],
        **exp
    })

precomputed_df = pd.DataFrame(precomputed)

precomputed_df.to_json('models/IF_SHAP.json', orient='records', indent=2)
precomputed_df.to_pickle('models/IF_SHAP.pkl')

print(f"Precomputed SHAP explanations untuk {len(precomputed_df)} gempa anomali berhasil disimpan.")
print("File: models/IF_SHAP.json / .pkl")

## 9. Terjemahan SHAP ke Bahasa Indonesia Awam

In [ ]:
def explain_to_indonesian(explanation, feature_labels=None):
    """Mengubah hasil explain_anomaly() menjadi kalimat Bahasa Indonesia mudah dipahami."""
    if feature_labels is None:
        feature_labels = {
            "mag": "magnitudo",
            "depth": "kedalaman",
            "latitude": "posisi lintang",
            "longitude": "posisi bujur",
        }

    ranking = explanation["ranking"]
    total_abs = sum(abs(v) for _, v in ranking)

    if total_abs == 0:
        return "Tidak ada fitur yang berkontribusi signifikan terhadap skor anomali ini."

    kontribusi_persen = [(feat, abs(val) / total_abs * 100, val) for feat, val in ranking]

    kalimat_bagian = []
    for i, (feat, persen, val) in enumerate(kontribusi_persen[:3]):
        label = feature_labels.get(feat, feat)
        arah = "tidak biasa" if val < 0 else "masih dalam batas wajar namun tercatat"
        if i == 0:
            kalimat_bagian.append(f"{label} yang {arah} adalah faktor utama (kontribusi {persen:.0f}%)")
        else:
            kalimat_bagian.append(f"{label} yang {arah} (kontribusi {persen:.0f}%)")

    kalimat = f"Gempa ini dianggap {'anomali' if explanation['is_anomaly'] else 'normal'} karena " \
              + kalimat_bagian[0]
    if len(kalimat_bagian) > 1:
        kalimat += ", diikuti oleh " + ", ".join(kalimat_bagian[1:]) + "."
    else:
        kalimat += "."

    return kalimat


print(explain_to_indonesian(contoh))

## 10. Visualisasi Global: Feature Importance dari SHAP (Top Anomali)

Melihat fitur mana yang secara rata-rata paling sering menjadi penyebab anomali di seluruh histori.

In [ ]:
import matplotlib.pyplot as plt

shap_matrix = pd.DataFrame(list(precomputed_df['shap_values']))
mean_abs_shap = shap_matrix.abs().mean().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
mean_abs_shap.plot(kind='barh', color='crimson')
plt.gca().invert_yaxis()
plt.title('Rata-rata |SHAP Value| per Fitur (Seluruh Gempa Anomali)', fontsize=13)
plt.xlabel('Rata-rata Kontribusi Absolut terhadap Skor Anomali')
plt.tight_layout()
plt.show()

mean_abs_shap

## 11. Catatan Produksi (FastAPI)

- **Histori (data lama)**: cukup baca `models/IF_SHAP.json` / `.pkl` dari Cell 8 di backend — **tidak perlu install `shap` di server produksi**.
- **Gempa baru real-time**: dua opsi:
  1. Precompute ulang secara periodik lewat job terpisah (yang punya `shap` terinstal), lalu simpan hasil barunya ke file/DB yang dibaca API — direkomendasikan untuk throughput tinggi.
  2. Panggil `explain_anomaly()` langsung di endpoint FastAPI — dengan 4 fitur dan background 200 sampel, cukup cepat (puluhan-ratusan ms per titik karena hanya 24 permutasi exact), aman untuk traffic rendah/demo skripsi.
